In [ ]:
# Mount Google Drive and initialize data preparation paths

from google.colab import drive
from pathlib import Path

import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/NLP04_Toxic_Comment_Classification')

CONFIG_DIR = PROJECT_ROOT / '00_config'
RAW_DIR = PROJECT_ROOT / '01_data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / '01_data' / 'processed'
SPLITS_DIR = PROJECT_ROOT / '01_data' / 'splits'
TOKENIZED_DIR = PROJECT_ROOT / '01_data' / 'tokenized'

NOTEBOOK_DIR = PROJECT_ROOT / '02_notebooks' / '02_data_preparation'
RESULTS_DIR = NOTEBOOK_DIR / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
TABLES_DIR = RESULTS_DIR / 'tables'

assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

for folder in [
    PROCESSED_DIR, SPLITS_DIR, TOKENIZED_DIR,
    RESULTS_DIR, FIGURES_DIR, TABLES_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Load shared configuration and initialize seeds
with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']

random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid')

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

print("Notebook:", NOTEBOOK_DIR.name)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Splits directory:", SPLITS_DIR)
print("Tokenized data directory:", TOKENIZED_DIR)
print("Results directory:", RESULTS_DIR)
print("Target labels:", LABELS)
print("Random seed:", SEED)

In [ ]:
# Install and verify dependencies required for data preparation

import sys
import subprocess
import importlib.util
import importlib.metadata as metadata

if importlib.util.find_spec('iterstrat') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '-q', 'iterative-stratification'
    ])

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from transformers import AutoTokenizer

# Record dependency versions
packages = [
    'numpy',
    'pandas',
    'scikit-learn',
    'iterative-stratification',
    'transformers',
    'tokenizers',
    'pyarrow'
]

dependency_df = pd.DataFrame({
    'Package': packages,
    'Version': [metadata.version(package) for package in packages]
})

display(dependency_df)

dependency_df.to_csv(
    TABLES_DIR / 'dependency_versions.csv',
    index=False
)

print("Multilabel stratification available.")
print("Transformer tokenizers available.")

In [ ]:
# Verify raw dataset fingerprints and load the original datasets

import hashlib

manifest_path = (
    PROJECT_ROOT / '02_notebooks' / '00_project_setup'
    / 'results' / 'tables' / 'dataset_manifest.csv'
)

manifest_df = pd.read_csv(manifest_path)

for _, row in manifest_df.iterrows():
    path = RAW_DIR / row['filename']
    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            sha256.update(chunk)

    assert sha256.hexdigest() == row['sha256'], \
        f"Integrity check failed: {path.name}"

    print(f"{path.name}: Integrity verified")

# Load original datasets
train_df = pd.read_csv(RAW_DIR / 'train.csv')
test_df = pd.read_csv(RAW_DIR / 'test.csv')
test_labels_df = pd.read_csv(RAW_DIR / 'test_labels.csv')

# Verify IDs and label validity
assert train_df['id'].is_unique
assert test_df['id'].is_unique
assert test_labels_df['id'].is_unique

assert set(test_df['id']) == set(test_labels_df['id'])
assert train_df[LABELS].isin([0, 1]).all().all()
assert test_labels_df[LABELS].isin([-1, 0, 1]).all().all()

print("\nDataset dimensions")
print("Training data:", train_df.shape)
print("Test data:", test_df.shape)
print("Test labels:", test_labels_df.shape)

print("\nDataset verification completed.")

In [ ]:
# Merge test comments with annotations and retain fully labelled records

test_merged = test_df.merge(
    test_labels_df,
    on='id',
    how='inner',
    validate='one_to_one'
)

valid_mask = test_merged[LABELS].isin([0, 1]).all(axis=1)
unlabelled_mask = test_merged[LABELS].eq(-1).all(axis=1)

test_valid_df = (
    test_merged.loc[valid_mask, ['id', 'comment_text'] + LABELS]
    .reset_index(drop=True)
    .copy()
)

# Verify the official evaluation partition
assert len(test_merged) == len(test_df)
assert len(test_valid_df) == 63978
assert test_valid_df['id'].is_unique
assert test_valid_df[LABELS].isin([0, 1]).all().all()
assert test_valid_df['comment_text'].notna().all()
assert test_valid_df['comment_text'].str.strip().ne('').all()

# Save test preparation statistics
test_preparation_df = pd.DataFrame({
    'Metric': [
        'Original test comments',
        'Fully labelled test comments',
        'Unlabelled test comments',
        'Partially labelled test comments',
        'Final evaluation comments'
    ],
    'Count': [
        len(test_merged),
        valid_mask.sum(),
        unlabelled_mask.sum(),
        (~(valid_mask | unlabelled_mask)).sum(),
        len(test_valid_df)
    ]
})

display(test_preparation_df)

test_preparation_df.to_csv(
    TABLES_DIR / 'test_preparation_summary.csv',
    index=False
)

print("Final test dataset shape:", test_valid_df.shape)
print("Official test partition prepared successfully.")

In [ ]:
# Detect normalized text overlaps and prepare the development pool

def normalize_text(series):
    return (
        series.str.lower()
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

train_norm = normalize_text(train_df['comment_text'])
test_valid_norm = normalize_text(test_valid_df['comment_text'])

# Identify overlapping training records
overlap_mask = train_norm.isin(test_valid_norm)

removed_df = train_df.loc[overlap_mask].copy()

development_df = (
    train_df.loc[~overlap_mask]
    .reset_index(drop=True)
    .copy()
)

# Verify consistency with the EDA overlap audit
eda_overlap_path = (
    PROJECT_ROOT / '02_notebooks' / '01_dataset_eda'
    / 'results' / 'tables' / 'overlapping_training_ids.csv'
)

eda_overlap_ids = pd.read_csv(eda_overlap_path)

assert set(removed_df['id']) == set(eda_overlap_ids['id'])

# Verify that detected overlaps have been eliminated
development_norm = normalize_text(development_df['comment_text'])

remaining_overlap = development_norm.isin(test_valid_norm).sum()

assert remaining_overlap == 0
assert len(development_df) + len(removed_df) == len(train_df)
assert development_df['id'].is_unique

# Summarize leakage prevention
leakage_summary_df = pd.DataFrame({
    'Metric': [
        'Original training comments',
        'Overlapping training comments removed',
        'Final development pool',
        'Official labelled test comments',
        'Remaining normalized train-test overlaps'
    ],
    'Count': [
        len(train_df),
        len(removed_df),
        len(development_df),
        len(test_valid_df),
        remaining_overlap
    ]
})

display(leakage_summary_df)

# Record the impact of overlap removal on label distributions
removal_impact_df = pd.DataFrame({
    'Label': LABELS,
    'Original positives': train_df[LABELS].sum().values,
    'Removed positives': removed_df[LABELS].sum().values,
    'Remaining positives': development_df[LABELS].sum().values
})

display(removal_impact_df)

# Save leakage prevention records
leakage_summary_df.to_csv(
    TABLES_DIR / 'leakage_removal_summary.csv', index=False
)

removal_impact_df.to_csv(
    TABLES_DIR / 'leakage_removal_label_impact.csv', index=False
)

removed_df[['id'] + LABELS].to_csv(
    TABLES_DIR / 'excluded_training_records.csv', index=False
)

print("\nDevelopment pool shape:", development_df.shape)
print("Normalized train-test overlap removed successfully.")

In [ ]:
# Group normalized duplicate comments before dataset splitting

development_df['group_id'] = pd.factorize(
    development_norm, sort=False
)[0]

# Create group-level labels for multilabel stratification
group_df = (
    development_df.groupby('group_id')[LABELS]
    .max()
)

group_df['group_size'] = (
    development_df.groupby('group_id').size()
)

group_df = group_df.reset_index()

# Identify groups containing conflicting annotations
group_min = development_df.groupby('group_id')[LABELS].min()
group_max = development_df.groupby('group_id')[LABELS].max()

conflict_mask = group_min.ne(group_max).any(axis=1)

# Summarize duplicate groups
group_summary_df = pd.DataFrame({
    'Metric': [
        'Development comments',
        'Total unique groups',
        'Single-comment groups',
        'Duplicate groups',
        'Comments within duplicate groups',
        'Excess duplicate records',
        'Groups with conflicting annotations',
        'Largest group size'
    ],
    'Count': [
        len(development_df),
        len(group_df),
        group_df['group_size'].eq(1).sum(),
        group_df['group_size'].gt(1).sum(),
        group_df.loc[group_df['group_size'] > 1, 'group_size'].sum(),
        len(development_df) - len(group_df),
        conflict_mask.sum(),
        group_df['group_size'].max()
    ]
})

display(group_summary_df)

# Verify group assignments
assert development_df['group_id'].notna().all()
assert group_df['group_id'].is_unique
assert group_df['group_size'].sum() == len(development_df)
assert len(group_df) == development_norm.nunique()

group_summary_df.to_csv(
    TABLES_DIR / 'duplicate_group_summary.csv',
    index=False
)

print("Group-level dataset shape:", group_df.shape)
print("All comments assigned to groups successfully.")

In [ ]:
# Perform group-aware multilabel stratified train-validation splitting

TRAIN_RATIO = config['split']['train_ratio']
VAL_RATIO = config['split']['validation_ratio']

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=VAL_RATIO,
    random_state=SEED
)

# Stratify using group-level labels
X_groups = group_df[['group_id']].values
y_groups = group_df[LABELS].values

train_idx, val_idx = next(
    splitter.split(X_groups, y_groups)
)

train_group_ids = set(group_df.iloc[train_idx]['group_id'])
val_group_ids = set(group_df.iloc[val_idx]['group_id'])

# Assign complete groups to their respective partitions
train_full_df = (
    development_df[
        development_df['group_id'].isin(train_group_ids)
    ]
    .reset_index(drop=True)
    .copy()
)

val_df = (
    development_df[
        development_df['group_id'].isin(val_group_ids)
    ]
    .reset_index(drop=True)
    .copy()
)

# Verify partition integrity
assert train_group_ids.isdisjoint(val_group_ids)

assert len(train_full_df) + len(val_df) == len(development_df)

assert set(train_full_df['id']).isdisjoint(val_df['id'])

assert train_group_ids | val_group_ids == set(group_df['group_id'])

# Summarize the resulting partitions
split_summary_df = pd.DataFrame({
    'Partition': ['Development pool', 'Training', 'Validation'],
    'Comments': [
        len(development_df),
        len(train_full_df),
        len(val_df)
    ],
    'Unique Groups': [
        len(group_df),
        len(train_group_ids),
        len(val_group_ids)
    ],
    'Percentage': [
        100,
        len(train_full_df) / len(development_df) * 100,
        len(val_df) / len(development_df) * 100
    ]
})

display(split_summary_df.round(3))

split_summary_df.to_csv(
    TABLES_DIR / 'train_validation_split_summary.csv',
    index=False
)

print("Training dataset shape:", train_full_df.shape)
print("Validation dataset shape:", val_df.shape)
print("Shared group IDs:", len(train_group_ids & val_group_ids))
print("Group-aware multilabel splitting completed.")

In [ ]:
# Verify partition integrity and check for cross-partition leakage

partitions = {
    'Training': train_full_df,
    'Validation': val_df,
    'Test': test_valid_df
}

partition_checks = []

# Verify individual partitions
for name, df in partitions.items():

    partition_checks.extend([
        {
            'Check': f'{name} - Duplicate IDs',
            'Count': df['id'].duplicated().sum()
        },
        {
            'Check': f'{name} - Missing values',
            'Count': df.isna().sum().sum()
        },
        {
            'Check': f'{name} - Empty comments',
            'Count': df['comment_text'].str.strip().eq('').sum()
        },
        {
            'Check': f'{name} - Invalid labels',
            'Count': (~df[LABELS].isin([0, 1])).sum().sum()
        }
    ])

# Check cross-partition ID and normalized text overlaps
from itertools import combinations

partition_ids = {
    name: set(df['id'])
    for name, df in partitions.items()
}

partition_texts = {
    name: set(normalize_text(df['comment_text']))
    for name, df in partitions.items()
}

for first, second in combinations(partitions.keys(), 2):

    partition_checks.extend([
        {
            'Check': f'{first}-{second} ID overlap',
            'Count': len(partition_ids[first] & partition_ids[second])
        },
        {
            'Check': f'{first}-{second} normalized text overlap',
            'Count': len(partition_texts[first] & partition_texts[second])
        }
    ])

# Verify group separation and development pool coverage
partition_checks.extend([
    {
        'Check': 'Shared training-validation groups',
        'Count': len(train_group_ids & val_group_ids)
    },
    {
        'Check': 'Development partition ID mismatch',
        'Count': len(
            (partition_ids['Training'] | partition_ids['Validation'])
            ^ set(development_df['id'])
        )
    },
    {
        'Check': 'Excluded training IDs retained in development',
        'Count': len(
            set(removed_df['id']) & set(development_df['id'])
        )
    }
])

integrity_df = pd.DataFrame(partition_checks)

integrity_df['Status'] = np.where(
    integrity_df['Count'].eq(0), 'Passed', 'Failed'
)

display(integrity_df)

integrity_df.to_csv(
    TABLES_DIR / 'partition_integrity_checks.csv',
    index=False
)

assert integrity_df['Count'].eq(0).all(), "Partition integrity check failed"

print("Total integrity checks:", len(integrity_df))
print("Passed checks:", integrity_df['Status'].eq('Passed').sum())
print("Failed checks:", integrity_df['Status'].eq('Failed').sum())
print("Partition integrity verified successfully.")

In [ ]:
# Compare label frequencies and prevalence across development partitions

split_label_records = []

for label in LABELS:

    dev_positive = development_df[label].sum()
    train_positive = train_full_df[label].sum()
    val_positive = val_df[label].sum()

    dev_rate = development_df[label].mean() * 100
    train_rate = train_full_df[label].mean() * 100
    val_rate = val_df[label].mean() * 100

    split_label_records.append({
        'Label': label,
        'Development Positives': dev_positive,
        'Training Positives': train_positive,
        'Validation Positives': val_positive,
        'Development (%)': dev_rate,
        'Training (%)': train_rate,
        'Validation (%)': val_rate,
        'Train-Val Difference (pp)': abs(train_rate - val_rate),
        'Validation Share of Positives (%)':
            val_positive / dev_positive * 100
    })

split_label_df = pd.DataFrame(split_label_records)

display(split_label_df.round(4))

split_label_df.to_csv(
    TABLES_DIR / 'split_label_distribution.csv',
    index=False
)

# Verify label coverage and positive count preservation

assert (split_label_df['Training Positives'] > 0).all()
assert (split_label_df['Validation Positives'] > 0).all()

assert (
    split_label_df['Training Positives']
    + split_label_df['Validation Positives']
    == split_label_df['Development Positives']
).all()

# Visualize label prevalence across partitions

plot_df = split_label_df.set_index('Label')[
    ['Development (%)', 'Training (%)', 'Validation (%)']
]

fig, ax = plt.subplots(figsize=(13, 6))

plot_df.plot(
    kind='bar',
    color=['#9E9E9E', '#4C78A8', '#B279A2'],
    width=0.75,
    ax=ax
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.2f',
        padding=3,
        fontsize=8
    )

ax.set_title('Label Distribution Across Development Partitions',
             fontsize=14)

ax.set_xlabel('Toxicity Label')
ax.set_ylabel('Positive Label Prevalence (%)')

ax.set_ylim(0, plot_df.to_numpy().max() * 1.25)

ax.legend([
    'Development pool',
    'Training',
    'Validation'
])

ax.tick_params(axis='x', rotation=0)
ax.grid(axis='x', visible=False)

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'split_label_distribution.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Compare clean, toxic and multilabel distributions across partitions

development_partitions = {
    'Development': development_df,
    'Training': train_full_df,
    'Validation': val_df
}

cardinality_records = []
distribution_records = []

for name, df in development_partitions.items():

    cardinality = df[LABELS].sum(axis=1)

    clean = int(cardinality.eq(0).sum())
    toxic = int(cardinality.gt(0).sum())
    single = int(cardinality.eq(1).sum())
    multi = int(cardinality.gt(1).sum())

    distribution_records.append({
        'Partition': name,
        'Total Comments': len(df),
        'Clean Comments': clean,
        'Toxic Comments': toxic,
        'Clean (%)': clean / len(df) * 100,
        'Toxic (%)': toxic / len(df) * 100,
        'Single-label Comments': single,
        'Multi-label Comments': multi,
        'Multi-label Among Toxic (%)': multi / toxic * 100,
        'Mean Label Cardinality': cardinality.mean(),
        'Label Density': cardinality.mean() / len(LABELS)
    })

    for k in range(len(LABELS) + 1):

        count = int(cardinality.eq(k).sum())

        cardinality_records.append({
            'Partition': name,
            'Number of Labels': k,
            'Count': count,
            'Percentage of All Comments': count / len(df) * 100,
            'Percentage of Toxic Comments':
                count / toxic * 100 if k > 0 else np.nan
        })

distribution_comparison_df = pd.DataFrame(distribution_records)
cardinality_comparison_df = pd.DataFrame(cardinality_records)

print("Clean and toxic distribution")
display(distribution_comparison_df.round(4))

# Compare label cardinality percentages

cardinality_pivot = cardinality_comparison_df.pivot(
    index='Number of Labels',
    columns='Partition',
    values='Percentage of All Comments'
)

cardinality_pivot = cardinality_pivot[
    ['Development', 'Training', 'Validation']
]

print("\nLabel cardinality distribution (%)")
display(cardinality_pivot.round(4))

# Verify preservation of cardinality counts

cardinality_counts = cardinality_comparison_df.pivot(
    index='Number of Labels',
    columns='Partition',
    values='Count'
)

assert (
    cardinality_counts['Training']
    + cardinality_counts['Validation']
    == cardinality_counts['Development']
).all()

# Save comparison tables

distribution_comparison_df.to_csv(
    TABLES_DIR / 'split_clean_toxic_comparison.csv',
    index=False
)

cardinality_comparison_df.to_csv(
    TABLES_DIR / 'split_label_cardinality_comparison.csv',
    index=False
)

print("\nLabel cardinality counts preserved across partitions.")

In [ ]:
# Estimate label representation across candidate training subset sizes

full_train_size = len(train_full_df)

candidate_sizes = [
    30000,
    40000,
    60000,
    80000,
    100000,
    full_train_size
]

train_prevalence = train_full_df[LABELS].mean()

sample_records = []

for size in candidate_sizes:

    record = {
        'Training Samples': size,
        'Data Retained (%)': size / full_train_size * 100,
        'Data Reduction (%)': (1 - size / full_train_size) * 100
    }

    for label in LABELS:
        record[label] = round(size * train_prevalence[label])

    sample_records.append(record)

sample_size_df = pd.DataFrame(sample_records)

print("Expected positive label counts under representative sampling")
display(sample_size_df.round(2))

sample_size_df.to_csv(
    TABLES_DIR / 'subsample_size_analysis.csv',
    index=False
)

# Visualize rare-label representation across sample sizes

rare_labels = ['severe_toxic', 'threat', 'identity_hate']

fig, ax = plt.subplots(figsize=(12, 6))

for label in rare_labels:

    ax.plot(
        sample_size_df['Training Samples'],
        sample_size_df[label],
        marker='o',
        markersize=7,
        linewidth=2,
        color=LABEL_COLORS[label],
        label=label
    )

    for size, count in zip(
        sample_size_df['Training Samples'],
        sample_size_df[label]
    ):
        ax.annotate(
            f'{count:,}',
            (size, count),
            xytext=(0, 9),
            textcoords='offset points',
            ha='center',
            fontsize=9
        )

ax.axvline(
    60000,
    color='grey',
    linestyle='--',
    linewidth=1.5,
    label='Proposed 60k subset'
)

ax.set_title(
    'Expected Rare-Label Representation Across Training Sizes',
    fontsize=14
)

ax.set_xlabel('Training Sample Size')
ax.set_ylabel('Expected Positive Comments')

ax.set_xticks(candidate_sizes)
ax.set_xticklabels([
    f'{size / 1000:.1f}k' if size == full_train_size
    else f'{size // 1000}k'
    for size in candidate_sizes
])

ax.set_ylim(0, sample_size_df[rare_labels].to_numpy().max() * 1.15)
ax.legend()
ax.grid(axis='x', visible=False)

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'subsample_size_rare_label_support.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Create a representative training subset using group-aware stratification

TARGET_SUBSET_SIZE = config['sampling']['initial_subset_size']

training_groups_df = (
    group_df[group_df['group_id'].isin(train_group_ids)]
    .reset_index(drop=True)
    .copy()
)

# Calculate the required number of groups
subset_ratio = TARGET_SUBSET_SIZE / len(train_full_df)

total_groups = len(training_groups_df)
target_group_count = int(round(total_groups * subset_ratio))
remaining_group_count = total_groups - target_group_count

# Explicitly specify both partition sizes
subset_splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    train_size=target_group_count,
    test_size=remaining_group_count,
    random_state=SEED
)

subset_idx, unused_idx = next(
    subset_splitter.split(
        training_groups_df[['group_id']].values,
        training_groups_df[LABELS].values
    )
)

subset_group_ids = set(
    training_groups_df.iloc[subset_idx]['group_id']
)

# Retain complete duplicate groups
train_subset_df = (
    train_full_df[
        train_full_df['group_id'].isin(subset_group_ids)
    ]
    .reset_index(drop=True)
    .copy()
)

# Verify subset integrity
assert subset_group_ids.issubset(train_group_ids)
assert subset_group_ids.isdisjoint(val_group_ids)

assert set(train_subset_df['id']).issubset(
    set(train_full_df['id'])
)

assert train_subset_df['id'].is_unique
assert train_subset_df[LABELS].sum().gt(0).all()

original_group_sizes = train_full_df.groupby('group_id').size()
subset_group_sizes = train_subset_df.groupby('group_id').size()

assert subset_group_sizes.equals(
    original_group_sizes.loc[subset_group_sizes.index]
)

# Summarize the selected subset
subset_summary_df = pd.DataFrame({
    'Metric': [
        'Full training comments',
        'Target subset size',
        'Actual subset size',
        'Difference from target',
        'Selected groups',
        'Unused training comments',
        'Training data retained (%)',
        'Training data reduction (%)'
    ],
    'Value': [
        len(train_full_df),
        TARGET_SUBSET_SIZE,
        len(train_subset_df),
        len(train_subset_df) - TARGET_SUBSET_SIZE,
        len(subset_group_ids),
        len(train_full_df) - len(train_subset_df),
        len(train_subset_df) / len(train_full_df) * 100,
        (1 - len(train_subset_df) / len(train_full_df)) * 100
    ]
})

display(subset_summary_df.round(4))

subset_summary_df.to_csv(
    TABLES_DIR / 'training_subset_summary.csv',
    index=False
)

print("Training subset shape:", train_subset_df.shape)
print("Validation dataset shape:", val_df.shape)
print("Official test dataset shape:", test_valid_df.shape)
print("Training subset integrity verified.")

In [ ]:
# Compare label representation in the full training data and sampled subset

subset_label_records = []

for label in LABELS:

    full_positive = int(train_full_df[label].sum())
    subset_positive = int(train_subset_df[label].sum())

    full_rate = train_full_df[label].mean() * 100
    subset_rate = train_subset_df[label].mean() * 100

    subset_label_records.append({
        'Label': label,
        'Full Training Positives': full_positive,
        'Subset Positives': subset_positive,
        'Expected Subset Positives': round(
            len(train_subset_df) * full_rate / 100
        ),
        'Full Training (%)': full_rate,
        'Subset (%)': subset_rate,
        'Absolute Difference (pp)': abs(full_rate - subset_rate),
        'Positives Retained (%)': subset_positive / full_positive * 100
    })

subset_label_df = pd.DataFrame(subset_label_records)

display(subset_label_df.round(4))

subset_label_df.to_csv(
    TABLES_DIR / 'subset_label_distribution.csv',
    index=False
)

# Compare clean and multilabel comment distributions

subset_distribution_records = []

for name, df in {
    'Full Training': train_full_df,
    'Representative Subset': train_subset_df
}.items():

    cardinality = df[LABELS].sum(axis=1)

    clean = int(cardinality.eq(0).sum())
    toxic = int(cardinality.gt(0).sum())
    multi = int(cardinality.gt(1).sum())

    subset_distribution_records.append({
        'Partition': name,
        'Comments': len(df),
        'Clean Comments': clean,
        'Toxic Comments': toxic,
        'Clean (%)': clean / len(df) * 100,
        'Toxic (%)': toxic / len(df) * 100,
        'Multi-label Among Toxic (%)': multi / toxic * 100,
        'Mean Label Cardinality': cardinality.mean(),
        'Label Density': cardinality.mean() / len(LABELS)
    })

subset_distribution_df = pd.DataFrame(subset_distribution_records)

display(subset_distribution_df.round(4))

subset_distribution_df.to_csv(
    TABLES_DIR / 'subset_clean_toxic_comparison.csv',
    index=False
)

# Visualize label prevalence before and after subsampling

plot_df = subset_label_df.set_index('Label')[
    ['Full Training (%)', 'Subset (%)']
]

fig, ax = plt.subplots(figsize=(12, 6))

plot_df.plot(
    kind='bar',
    color=['#4C78A8', '#54A24B'],
    width=0.65,
    ax=ax
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.3f',
        padding=3,
        fontsize=9
    )

ax.set_title(
    'Label Distribution Before and After Representative Subsampling',
    fontsize=14
)

ax.set_xlabel('Toxicity Label')
ax.set_ylabel('Positive Label Prevalence (%)')
ax.set_ylim(0, plot_df.to_numpy().max() * 1.2)
ax.tick_params(axis='x', rotation=0)
ax.legend(['Full Training', 'Representative Subset'])
ax.grid(axis='x', visible=False)

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'subset_label_distribution_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

print("Maximum label prevalence difference (pp):",
      round(subset_label_df['Absolute Difference (pp)'].max(), 4))

In [ ]:
# Create a toxicity-enriched subset while preserving all positive examples

ENRICHED_TARGET_SIZE = len(train_subset_df)

# Identify groups containing positive labels
positive_groups = training_groups_df[
    training_groups_df[LABELS].sum(axis=1).gt(0)
]

clean_groups = training_groups_df[
    training_groups_df[LABELS].sum(axis=1).eq(0)
].copy()

positive_group_ids = set(positive_groups['group_id'])

# Retain complete groups containing positive comments
mandatory_count = int(positive_groups['group_size'].sum())

remaining_budget = ENRICHED_TARGET_SIZE - mandatory_count

assert remaining_budget >= 0

# Randomly select clean-only groups
clean_groups = clean_groups.sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

cumulative_size = clean_groups['group_size'].cumsum()

selected_clean_groups = clean_groups[
    cumulative_size <= remaining_budget
]

selected_clean_ids = set(selected_clean_groups['group_id'])

# Fill any remaining positions using unused singleton groups
remaining_slots = remaining_budget - int(
    selected_clean_groups['group_size'].sum()
)

if remaining_slots > 0:

    available_singletons = clean_groups[
        ~clean_groups['group_id'].isin(selected_clean_ids)
        & clean_groups['group_size'].eq(1)
    ]

    assert len(available_singletons) >= remaining_slots

    selected_clean_ids.update(
        available_singletons.head(remaining_slots)['group_id']
    )

# Construct the enriched training subset
enriched_group_ids = positive_group_ids | selected_clean_ids

train_enriched_df = (
    train_full_df[
        train_full_df['group_id'].isin(enriched_group_ids)
    ]
    .reset_index(drop=True)
    .copy()
)

# Verify subset size, positive retention and group integrity
assert len(train_enriched_df) == len(train_subset_df)

assert set(train_enriched_df['id']).issubset(
    set(train_full_df['id'])
)

assert enriched_group_ids.isdisjoint(val_group_ids)

assert train_enriched_df[LABELS].sum().equals(
    train_full_df[LABELS].sum()
)

assert train_enriched_df['id'].is_unique

enriched_group_sizes = train_enriched_df.groupby('group_id').size()

assert enriched_group_sizes.equals(
    train_full_df.groupby('group_id').size()
    .loc[enriched_group_sizes.index]
)

# Summarize the enriched subset
enriched_cardinality = train_enriched_df[LABELS].sum(axis=1)

enriched_summary_df = pd.DataFrame({
    'Metric': [
        'Total comments',
        'Clean comments',
        'Toxic comments',
        'Clean (%)',
        'Toxic (%)',
        'Selected groups',
        'Positive labels retained (%)'
    ],
    'Value': [
        len(train_enriched_df),
        enriched_cardinality.eq(0).sum(),
        enriched_cardinality.gt(0).sum(),
        enriched_cardinality.eq(0).mean() * 100,
        enriched_cardinality.gt(0).mean() * 100,
        len(enriched_group_ids),
        100.0
    ]
})

display(enriched_summary_df.round(4))

enriched_summary_df.to_csv(
    TABLES_DIR / 'enriched_subset_summary.csv',
    index=False
)

print("Enriched subset shape:", train_enriched_df.shape)
print("All positive annotations retained.")
print("Group integrity verified.")

In [ ]:
# Compare dataset composition, label prevalence and positive retention

training_datasets = {
    'Full Training': train_full_df,
    'Representative': train_subset_df,
    'Toxicity-Enriched': train_enriched_df
}

composition_records = []
label_records = []

for name, df in training_datasets.items():

    cardinality = df[LABELS].sum(axis=1)

    composition_records.append({
        'Dataset': name,
        'Total Comments': len(df),
        'Clean Comments': cardinality.eq(0).sum(),
        'Toxic Comments': cardinality.gt(0).sum(),
        'Clean (%)': cardinality.eq(0).mean() * 100,
        'Toxic (%)': cardinality.gt(0).mean() * 100,
        'Mean Label Cardinality': cardinality.mean()
    })

    for label in LABELS:

        positive = int(df[label].sum())
        full_positive = int(train_full_df[label].sum())

        label_records.append({
            'Dataset': name,
            'Label': label,
            'Positive Comments': positive,
            'Negative Comments': len(df) - positive,
            'Prevalence (%)': positive / len(df) * 100,
            'Positives Retained (%)': positive / full_positive * 100
        })

sampling_composition_df = pd.DataFrame(composition_records)
sampling_label_df = pd.DataFrame(label_records)

print("Training dataset composition")
display(sampling_composition_df.round(4))

print("\nPositive label counts")
display(sampling_label_df.pivot(
    index='Label',
    columns='Dataset',
    values='Positive Comments'
).reindex(LABELS))

print("\nLabel prevalence (%)")
display(sampling_label_df.pivot(
    index='Label',
    columns='Dataset',
    values='Prevalence (%)'
).reindex(LABELS).round(4))

# Save sampling comparison tables

sampling_composition_df.to_csv(
    TABLES_DIR / 'sampling_strategy_composition.csv',
    index=False
)

sampling_label_df.to_csv(
    TABLES_DIR / 'sampling_strategy_label_comparison.csv',
    index=False
)

# Compare dataset composition and rare-label representation

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Clean and toxic proportions

composition_plot = sampling_composition_df.set_index('Dataset')[
    ['Clean (%)', 'Toxic (%)']
]

composition_plot.plot(
    kind='bar',
    stacked=True,
    color=['#4C78A8', '#E45756'],
    width=0.6,
    ax=axes[0]
)

for container in axes[0].containers:
    axes[0].bar_label(
        container,
        fmt='%.1f%%',
        label_type='center',
        color='white',
        fontsize=11
    )

axes[0].set_title('Training Dataset Composition')
axes[0].set_xlabel('')
axes[0].set_ylabel('Percentage of Comments')
axes[0].set_ylim(0, 110)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['Clean', 'Toxic'], loc='upper right')
axes[0].grid(axis='x', visible=False)

# Rare-label positive counts

rare_labels = ['severe_toxic', 'threat', 'identity_hate']

rare_plot = sampling_label_df.pivot(
    index='Label',
    columns='Dataset',
    values='Positive Comments'
).loc[rare_labels, [
    'Full Training',
    'Representative',
    'Toxicity-Enriched'
]]

rare_plot.plot(
    kind='bar',
    color=['#4C78A8', '#54A24B', '#F58518'],
    width=0.75,
    ax=axes[1]
)

for container in axes[1].containers:
    axes[1].bar_label(
        container,
        fmt='%.0f',
        padding=3,
        fontsize=9
    )

axes[1].set_title('Rare-Label Representation')
axes[1].set_xlabel('Toxicity Label')
axes[1].set_ylabel('Positive Comments')
axes[1].set_ylim(0, rare_plot.to_numpy().max() * 1.25)
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='x', visible=False)

fig.suptitle('Comparison of Training Sampling Strategies', fontsize=16)

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'sampling_strategy_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Initialize pretrained tokenizers and record their configurations

from transformers import AutoTokenizer
from huggingface_hub import HfApi

MODEL_CHECKPOINTS = {
    'DistilBERT': 'distilbert/distilbert-base-uncased',
    'RoBERTa': 'FacebookAI/roberta-base'
}

tokenizers = {}
tokenizer_records = []

api = HfApi()

for model_name, checkpoint in MODEL_CHECKPOINTS.items():

    # Record the exact pretrained checkpoint revision
    revision = api.model_info(checkpoint).sha

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint,
        revision=revision,
        use_fast=True
    )

    tokenizers[model_name] = tokenizer

    tokenizer_records.append({
        'Model': model_name,
        'Checkpoint': checkpoint,
        'Revision': revision,
        'Tokenizer': tokenizer.__class__.__name__,
        'Vocabulary Size': tokenizer.vocab_size,
        'Maximum Supported Length': tokenizer.model_max_length,
        'Special Tokens': tokenizer.num_special_tokens_to_add(pair=False),
        'Padding Token': tokenizer.pad_token,
        'Unknown Token': tokenizer.unk_token,
        'Fast Tokenizer': tokenizer.is_fast
    })

    print(f"{model_name} tokenizer loaded successfully")

tokenizer_info_df = pd.DataFrame(tokenizer_records)

display(tokenizer_info_df)

tokenizer_info_df.to_csv(
    TABLES_DIR / 'tokenizer_information.csv',
    index=False
)

In [ ]:
# Calculate untruncated token lengths for both transformer tokenizers

from tqdm.auto import tqdm

texts = train_full_df['comment_text'].tolist()
BATCH_SIZE = 512

token_length_df = train_full_df[['id'] + LABELS].copy()

for model_name, tokenizer in tokenizers.items():

    lengths = []

    for start in tqdm(
        range(0, len(texts), BATCH_SIZE),
        desc=f'{model_name} token lengths'
    ):

        batch = texts[start:start + BATCH_SIZE]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            return_length=True,
            verbose=False
        )

        lengths.extend(encoded['length'])

    assert len(lengths) == len(train_full_df)

    token_length_df[f'{model_name}_tokens'] = lengths

# Summarize token length statistics

token_stats = []

for model_name in tokenizers:

    lengths = token_length_df[f'{model_name}_tokens']

    token_stats.append({
        'Model': model_name,
        'Comments': len(lengths),
        'Minimum': lengths.min(),
        'Mean': lengths.mean(),
        'Median': lengths.median(),
        'Standard Deviation': lengths.std(),
        '75th Percentile': lengths.quantile(0.75),
        '90th Percentile': lengths.quantile(0.90),
        '95th Percentile': lengths.quantile(0.95),
        '99th Percentile': lengths.quantile(0.99),
        'Maximum': lengths.max()
    })

token_stats_df = pd.DataFrame(token_stats)

display(token_stats_df.round(2))

# Save token lengths and summary statistics

token_length_df.to_parquet(
    TOKENIZED_DIR / 'training_token_lengths.parquet',
    index=False
)

token_stats_df.to_csv(
    TABLES_DIR / 'token_length_statistics.csv',
    index=False
)

print("Comments analysed:", len(token_length_df))
print("Tokenizers analysed:", len(tokenizers))
print("Token length analysis completed.")

In [ ]:
# Visualize token length distributions for DistilBERT and RoBERTa

MODEL_COLORS = {
    'DistilBERT': '#4C78A8',
    'RoBERTa': '#F58518'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

upper_limit = int(np.ceil(max(
    token_length_df['DistilBERT_tokens'].quantile(0.99),
    token_length_df['RoBERTa_tokens'].quantile(0.99)
) / 100) * 100)

bins = np.linspace(0, upper_limit, 81)

for ax, model_name in zip(axes, MODEL_COLORS):

    lengths = token_length_df[f'{model_name}_tokens']

    ax.hist(
        lengths,
        bins=bins,
        color=MODEL_COLORS[model_name],
        alpha=0.8,
        edgecolor='white',
        linewidth=0.3
    )

    ax.axvline(
        128, color='#E45756', linestyle='--',
        linewidth=1.5, label='128 tokens'
    )

    ax.axvline(
        256, color='#54A24B', linestyle='--',
        linewidth=1.5, label='256 tokens'
    )

    ax.axvline(
        512, color='#B279A2', linestyle='--',
        linewidth=1.5, label='512 tokens'
    )

    ax.set_title(
        f'{model_name} Token Length Distribution',
        fontsize=13
    )

    ax.set_xlabel('Number of Tokens')
    ax.set_ylabel('Number of Comments')
    ax.set_xlim(0, upper_limit)
    ax.legend()
    ax.grid(axis='x', visible=False)

fig.suptitle(
    'Transformer Token Length Distributions',
    fontsize=16
)

fig.text(
    0.5, 0.01,
    f'Display limited to {upper_limit} tokens for readability. '
    'Token counts include special tokens.',
    ha='center',
    fontsize=10
)

sns.despine()
plt.tight_layout(rect=[0, 0.04, 1, 0.95])

plt.savefig(
    FIGURES_DIR / 'transformer_token_length_distributions.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Evaluate truncation and token retention at different sequence lengths

MAX_LENGTH_CANDIDATES = [64, 128, 192, 256, 384, 512]

truncation_records = []

for model_name, tokenizer in tokenizers.items():

    lengths = token_length_df[f'{model_name}_tokens'].to_numpy()
    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)

    content_lengths = lengths - special_tokens

    for max_length in MAX_LENGTH_CANDIDATES:

        truncated_mask = lengths > max_length
        retained_lengths = np.minimum(lengths, max_length)

        retained_content = retained_lengths - special_tokens

        removed_tokens = content_lengths - retained_content

        truncation_records.append({
            'Model': model_name,
            'Max Length': max_length,
            'Total Comments': len(lengths),
            'Truncated Comments': truncated_mask.sum(),
            'Truncation Rate (%)': truncated_mask.mean() * 100,
            'Fully Retained Comments (%)':
                (~truncated_mask).mean() * 100,
            'Total Content Tokens': content_lengths.sum(),
            'Content Tokens Removed': removed_tokens.sum(),
            'Token Retention (%)':
                retained_content.sum() / content_lengths.sum() * 100,
            'Mean Input Length': retained_lengths.mean(),
            'Mean Tokens Removed per Truncated Comment':
                removed_tokens[truncated_mask].mean()
                if truncated_mask.any() else 0
        })

truncation_df = pd.DataFrame(truncation_records)

display(truncation_df.round(2))

truncation_df.to_csv(
    TABLES_DIR / 'overall_truncation_analysis.csv',
    index=False
)

# Compare truncation rates and token retention

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics = [
    'Truncation Rate (%)',
    'Token Retention (%)'
]

titles = [
    'Comments Affected by Truncation',
    'Content Tokens Retained'
]

for ax, metric, title in zip(axes, metrics, titles):

    for model_name in tokenizers:

        model_df = truncation_df[
            truncation_df['Model'] == model_name
        ]

        ax.plot(
            model_df['Max Length'],
            model_df[metric],
            marker='o',
            markersize=7,
            linewidth=2,
            color=MODEL_COLORS[model_name],
            label=model_name
        )

    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Maximum Sequence Length')
    ax.set_ylabel(metric)
    ax.set_xticks(MAX_LENGTH_CANDIDATES)
    ax.legend()

axes[0].set_ylim(
    0, truncation_df['Truncation Rate (%)'].max() * 1.15
)

axes[1].set_ylim(0, 105)

fig.suptitle(
    'Impact of Maximum Sequence Length on Truncation',
    fontsize=16
)

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'sequence_length_truncation_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Evaluate truncation and token retention separately for each toxicity label

label_truncation_records = []

token_groups = {
    'clean': token_length_df[LABELS].sum(axis=1).eq(0),
    'any_toxic': token_length_df[LABELS].sum(axis=1).gt(0)
}

for label in LABELS:
    token_groups[label] = token_length_df[label].eq(1)

for model_name, tokenizer in tokenizers.items():

    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)

    for category, mask in token_groups.items():

        lengths = token_length_df.loc[
            mask, f'{model_name}_tokens'
        ].to_numpy()

        content_lengths = lengths - special_tokens

        for max_length in MAX_LENGTH_CANDIDATES:

            truncated_mask = lengths > max_length
            retained_lengths = np.minimum(lengths, max_length)
            retained_content = retained_lengths - special_tokens

            label_truncation_records.append({
                'Model': model_name,
                'Category': category,
                'Max Length': max_length,
                'Total Comments': len(lengths),
                'Truncated Comments': truncated_mask.sum(),
                'Truncation Rate (%)':
                    truncated_mask.mean() * 100,
                'Fully Retained Comments (%)':
                    (~truncated_mask).mean() * 100,
                'Token Retention (%)':
                    retained_content.sum() / content_lengths.sum() * 100,
                'Mean Original Length': lengths.mean(),
                'Mean Retained Length': retained_lengths.mean()
            })

label_truncation_df = pd.DataFrame(label_truncation_records)

label_truncation_df.to_csv(
    TABLES_DIR / 'label_specific_truncation.csv',
    index=False
)

# Display label-specific truncation rates

for model_name in tokenizers:

    print(f"\n{model_name} - Truncation Rate (%)")

    pivot = label_truncation_df[
        label_truncation_df['Model'] == model_name
    ].pivot(
        index='Category',
        columns='Max Length',
        values='Truncation Rate (%)'
    ).reindex(list(token_groups.keys()))

    display(pivot.round(2))

# Visualize label-specific truncation rates

fig, axes = plt.subplots(1, 2, figsize=(17, 7))

vmax = label_truncation_df['Truncation Rate (%)'].max()

for ax, model_name in zip(axes, tokenizers):

    pivot = label_truncation_df[
        label_truncation_df['Model'] == model_name
    ].pivot(
        index='Category',
        columns='Max Length',
        values='Truncation Rate (%)'
    ).reindex(list(token_groups.keys()))

    sns.heatmap(
        pivot,
        annot=True,
        fmt='.1f',
        cmap='YlOrRd',
        vmin=0,
        vmax=vmax,
        linewidths=0.5,
        cbar_kws={'label': 'Truncation Rate (%)'},
        ax=ax
    )

    ax.set_title(f'{model_name} Truncation Rates')
    ax.set_xlabel('Maximum Sequence Length')
    ax.set_ylabel('Comment Category')
    ax.tick_params(axis='y', rotation=0)

fig.suptitle(
    'Label-Specific Truncation Across Sequence Lengths',
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'label_specific_truncation_heatmaps.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Display content token retention by label

for model_name in tokenizers:

    print(f"\n{model_name} - Token Retention (%)")

    pivot = label_truncation_df[
        label_truncation_df['Model'] == model_name
    ].pivot(
        index='Category',
        columns='Max Length',
        values='Token Retention (%)'
    ).reindex(list(token_groups.keys()))

    display(pivot.round(2))

In [ ]:
# Compare token lengths and truncation across training sampling strategies

subset_token_records = []

for dataset_name, df in training_datasets.items():

    subset_lengths = token_length_df[
        token_length_df['id'].isin(df['id'])
    ]

    assert len(subset_lengths) == len(df)

    for model_name, tokenizer in tokenizers.items():

        lengths = subset_lengths[f'{model_name}_tokens'].to_numpy()

        special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
        content_lengths = lengths - special_tokens

        for max_length in MAX_LENGTH_CANDIDATES:

            truncated = lengths > max_length
            retained = np.minimum(lengths, max_length)
            retained_content = retained - special_tokens

            subset_token_records.append({
                'Dataset': dataset_name,
                'Model': model_name,
                'Max Length': max_length,
                'Comments': len(lengths),
                'Mean Original Length': lengths.mean(),
                'Median Original Length': np.median(lengths),
                'Truncated Comments': truncated.sum(),
                'Truncation Rate (%)': truncated.mean() * 100,
                'Token Retention (%)':
                    retained_content.sum() / content_lengths.sum() * 100,
                'Mean Input Length': retained.mean()
            })

subset_token_df = pd.DataFrame(subset_token_records)

subset_token_df.to_csv(
    TABLES_DIR / 'sampling_token_length_comparison.csv',
    index=False
)

# Display comparison at the main candidate sequence lengths

comparison_df = subset_token_df[
    subset_token_df['Max Length'].isin([128, 192, 256, 384, 512])
]

for model_name in tokenizers:

    print(f"\n{model_name} - Truncation Rate (%)")

    display(
        comparison_df[
            comparison_df['Model'] == model_name
        ].pivot(
            index='Dataset',
            columns='Max Length',
            values='Truncation Rate (%)'
        ).round(2)
    )

    print(f"\n{model_name} - Token Retention (%)")

    display(
        comparison_df[
            comparison_df['Model'] == model_name
        ].pivot(
            index='Dataset',
            columns='Max Length',
            values='Token Retention (%)'
        ).round(2)
    )

# Compare truncation at 256 tokens

plot_df = subset_token_df[
    subset_token_df['Max Length'] == 256
]

plot_pivot = plot_df.pivot(
    index='Dataset',
    columns='Model',
    values='Truncation Rate (%)'
).reindex(list(training_datasets.keys()))

fig, ax = plt.subplots(figsize=(10, 5.5))

plot_pivot.plot(
    kind='bar',
    color=[MODEL_COLORS[m] for m in plot_pivot.columns],
    width=0.65,
    ax=ax
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.2f%%',
        padding=4,
        fontsize=10
    )

ax.set_title('Truncation Across Training Subsets at 256 Tokens')
ax.set_xlabel('Training Dataset')
ax.set_ylabel('Truncation Rate (%)')
ax.set_ylim(0, plot_pivot.to_numpy().max() * 1.25)
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='x', visible=False)

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'sampling_strategy_truncation_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Record the initial sequence length decision and tokenizer configurations

MAX_LENGTH = 256

sequence_decision_df = subset_token_df[
    (subset_token_df['Dataset'] == 'Representative')
    & (subset_token_df['Max Length'].isin([256, 384, 512]))
].copy()

display(sequence_decision_df.round(2))

sequence_decision_df.to_csv(
    TABLES_DIR / 'sequence_length_decision.csv',
    index=False
)

# Update the shared project configuration
config['transformer'] = {
    'initial_max_length': MAX_LENGTH,
    'candidate_max_lengths': MAX_LENGTH_CANDIDATES,
    'padding_strategy': 'dynamic',
    'truncation': True,
    'truncation_side': 'right',
    'sequence_length_status': 'initial_selected',
    'ablation_lengths': [384, 512],
    'models': {}
}

for _, row in tokenizer_info_df.iterrows():

    config['transformer']['models'][row['Model']] = {
        'checkpoint': row['Checkpoint'],
        'revision': row['Revision'],
        'vocabulary_size': int(row['Vocabulary Size']),
        'special_tokens': int(row['Special Tokens']),
        'maximum_supported_length':
            int(row['Maximum Supported Length'])
    }

# Record the finalized dataset sizes
config['sampling'].update({
    'actual_subset_size': len(train_subset_df),
    'enriched_subset_size': len(train_enriched_df),
    'full_training_size': len(train_full_df),
    'primary_dataset': 'representative',
    'enriched_subset_status': 'optional_experiment',
    'status': 'initial_selected'
})

config['split'].update({
    'development_size': len(development_df),
    'training_size': len(train_full_df),
    'validation_size': len(val_df),
    'test_size': len(test_valid_df),
    'overlapping_training_records_removed': len(removed_df),
    'status': 'verified'
})

# Save updated configuration
with open(CONFIG_DIR / 'project_config.json', 'w') as file:
    json.dump(config, file, indent=4)

print("Initial maximum sequence length:", MAX_LENGTH)
print("Padding strategy: Dynamic padding")
print("Truncation strategy: Right truncation")
print("Representative subset:", len(train_subset_df))
print("Enriched subset:", len(train_enriched_df))
print("Full training:", len(train_full_df))
print("\nProject configuration updated.")

In [ ]:
# Save prepared datasets and verify the stored records

prepared_datasets = {
    'train_full.parquet': train_full_df,
    'train_representative.parquet': train_subset_df,
    'train_enriched.parquet': train_enriched_df,
    'validation.parquet': val_df,
    'test_official.parquet': test_valid_df
}

saved_records = []

for filename, df in prepared_datasets.items():

    path = PROCESSED_DIR / filename

    # Preserve existing datasets and check consistency
    if not path.exists():
        df.to_parquet(
            path,
            index=False,
            compression='snappy'
        )

    # Verify saved dataset against the original partition
    loaded_df = pd.read_parquet(path)

    pd.testing.assert_frame_equal(
        df.reset_index(drop=True),
        loaded_df.reset_index(drop=True)
    )

    saved_records.append({
        'Filename': filename,
        'Rows': len(loaded_df),
        'Columns': len(loaded_df.columns),
        'Size (MB)': path.stat().st_size / (1024 ** 2),
        'Verification': 'Passed'
    })

    print(f"{filename}: Saved and verified")

    del loaded_df

prepared_summary_df = pd.DataFrame(saved_records)

display(prepared_summary_df.round(2))

prepared_summary_df.to_csv(
    TABLES_DIR / 'prepared_dataset_summary.csv',
    index=False
)

print("\nTotal prepared datasets:", len(prepared_summary_df))
print("All prepared datasets verified successfully.")

In [ ]:
# Save partition IDs, group assignments and dataset fingerprints

split_records = []

for filename, df in prepared_datasets.items():

    partition_name = Path(filename).stem

    # Save partition membership
    manifest_columns = ['id']

    if 'group_id' in df.columns:
        manifest_columns.append('group_id')

    split_manifest = df[manifest_columns].copy()

    split_manifest['row_index'] = np.arange(len(df))

    split_manifest_path = (
        SPLITS_DIR / f'{partition_name}_ids.csv'
    )

    split_manifest.to_csv(split_manifest_path, index=False)

    # Calculate dataset fingerprint
    dataset_path = PROCESSED_DIR / filename

    sha256 = hashlib.sha256()

    with open(dataset_path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            sha256.update(chunk)

    split_records.append({
        'Partition': partition_name,
        'Filename': filename,
        'Rows': len(df),
        'Unique IDs': df['id'].nunique(),
        'Size Bytes': dataset_path.stat().st_size,
        'SHA256': sha256.hexdigest(),
        'Dataset Path': str(dataset_path.relative_to(PROJECT_ROOT)),
        'Manifest Path': str(
            split_manifest_path.relative_to(PROJECT_ROOT)
        )
    })

    # Verify saved partition membership and row order
    saved_manifest = pd.read_csv(
        split_manifest_path,
        dtype={'id': str}
    )

    pd.testing.assert_frame_equal(
        split_manifest,
        saved_manifest
    )

    print(f"{partition_name}: Manifest saved and verified")

# Save the shared dataset fingerprint registry
prepared_manifest_df = pd.DataFrame(split_records)

prepared_manifest_df.to_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv',
    index=False
)

display(prepared_manifest_df[
    ['Partition', 'Rows', 'Unique IDs', 'SHA256']
])

print("\nPartition manifests:", len(split_records))
print("Dataset fingerprint registry saved successfully.")

In [ ]:
# Update methodological decisions using verified data preparation results

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

decisions_df = pd.read_csv(DECISION_LOG_PATH)

updates = [
    {
        'decision_id': 'D001',
        'component': 'Data splitting',
        'decision': 'Group-aware 80:20 multilabel stratification',
        'justification': (
            '127628 training and 31911 validation comments. '
            'Maximum label prevalence difference is 0.0116 percentage points. '
            'All 21 partition integrity checks passed.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D002',
        'component': 'Test evaluation',
        'decision': 'Retain all 63978 fully labelled official test comments',
        'justification': (
            '89186 unlabelled records excluded. '
            'Official benchmark preserved for frozen final evaluation. '
            'Historical benchmark exposure acknowledged.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D003',
        'component': 'Subsampling',
        'decision': 'Representative subset of 59997 training comments',
        'justification': (
            'Reduces training sample count by 52.99%. '
            'Maximum label prevalence difference is 0.0047 percentage points. '
            'Retains 179 threat-positive examples. '
            'Full-data experiments will investigate the effect of data size.'
        ),
        'status': 'Initial selection'
    },
    {
        'decision_id': 'D007',
        'component': 'Maximum token length',
        'decision': '256 tokens for initial transformer experiments',
        'justification': (
            'Representative subset truncation rates are 6.86% for DistilBERT '
            'and 6.92% for RoBERTa. Content token retention is 82.38% '
            'and 81.86%, respectively. Longer sequence experiments '
            'will investigate rare-label performance.'
        ),
        'status': 'Initial selection'
    },
    {
        'decision_id': 'D009',
        'component': 'Leakage prevention',
        'decision': 'Remove 32 overlapping training records',
        'justification': (
            'Eliminates detected normalized-text overlap with the '
            'official labelled test dataset while preserving the benchmark.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D010',
        'component': 'Duplicate handling',
        'decision': 'Preserve complete normalized duplicate groups',
        'justification': (
            '233 duplicate groups remain in the development pool, '
            'including 41 groups with conflicting annotations. '
            'Original annotations are preserved without arbitrary relabelling.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D011',
        'component': 'Alternative sampling',
        'decision': 'Toxicity-enriched subset of 59997 comments',
        'justification': (
            'Retains all 12969 toxic training comments and all rare-label '
            'positives. Equal sample size enables a controlled comparison '
            'with representative sampling.'
        ),
        'status': 'Optional experiment'
    },
    {
        'decision_id': 'D012',
        'component': 'Text preprocessing',
        'decision': 'Preserve original comment text',
        'justification': (
            'EDA identified differences in capitalization and punctuation '
            'between clean and toxic comments. Normalized text is used '
            'only for duplicate detection and grouping.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D013',
        'component': 'Padding strategy',
        'decision': 'Dynamic batch-level padding',
        'justification': (
            'Median token length is approximately 52-53 tokens. '
            'Dynamic padding avoids unnecessary padding of short comments '
            'to the maximum sequence length.'
        ),
        'status': 'Selected'
    },
    {
        'decision_id': 'D014',
        'component': 'Sequence length ablation',
        'decision': 'Compare 256 tokens against 384 or 512 tokens',
        'justification': (
            'Severe toxic and threat comments exhibit substantial aggregate '
            'token loss at 256 tokens. Additional context may improve '
            'classification, subject to validation evidence and GPU resources.'
        ),
        'status': 'Planned'
    }
]

# Preserve the original decision log before updating

backup_path = CONFIG_DIR / 'project_decisions_initial.csv'

if not backup_path.exists():
    decisions_df.to_csv(backup_path, index=False)

# Update existing decisions and append new ones

for record in updates:

    mask = decisions_df['decision_id'].eq(record['decision_id'])

    if mask.any():
        for column, value in record.items():
            decisions_df.loc[mask, column] = value
    else:
        decisions_df = pd.concat(
            [decisions_df, pd.DataFrame([record])],
            ignore_index=True
        )

assert decisions_df['decision_id'].is_unique

decisions_df.to_csv(DECISION_LOG_PATH, index=False)

display(decisions_df[['decision_id', 'component', 'decision', 'status']])

print("Total methodological decisions:", len(decisions_df))
print("Decision log updated successfully.")

In [ ]:
# Verify data preparation artifacts and save the methodology summary

required_tables = [
    'dependency_versions.csv',
    'test_preparation_summary.csv',
    'leakage_removal_summary.csv',
    'leakage_removal_label_impact.csv',
    'excluded_training_records.csv',
    'duplicate_group_summary.csv',
    'train_validation_split_summary.csv',
    'partition_integrity_checks.csv',
    'split_label_distribution.csv',
    'split_clean_toxic_comparison.csv',
    'split_label_cardinality_comparison.csv',
    'subsample_size_analysis.csv',
    'training_subset_summary.csv',
    'subset_label_distribution.csv',
    'subset_clean_toxic_comparison.csv',
    'enriched_subset_summary.csv',
    'sampling_strategy_composition.csv',
    'sampling_strategy_label_comparison.csv',
    'tokenizer_information.csv',
    'token_length_statistics.csv',
    'overall_truncation_analysis.csv',
    'label_specific_truncation.csv',
    'sampling_token_length_comparison.csv',
    'sequence_length_decision.csv',
    'prepared_dataset_summary.csv'
]

required_figures = [
    'split_label_distribution.png',
    'subsample_size_rare_label_support.png',
    'subset_label_distribution_comparison.png',
    'sampling_strategy_comparison.png',
    'transformer_token_length_distributions.png',
    'sequence_length_truncation_comparison.png',
    'label_specific_truncation_heatmaps.png',
    'sampling_strategy_truncation_comparison.png'
]

required_shared_files = [
    *[PROCESSED_DIR / name for name in prepared_datasets],
    *[
        SPLITS_DIR / f'{Path(name).stem}_ids.csv'
        for name in prepared_datasets
    ],
    SPLITS_DIR / 'prepared_dataset_manifest.csv',
    TOKENIZED_DIR / 'training_token_lengths.parquet',
    CONFIG_DIR / 'project_config.json',
    CONFIG_DIR / 'project_decisions.csv',
    CONFIG_DIR / 'project_decisions_initial.csv'
]

# Verify all required artifacts

verification = []

artifact_groups = {
    'Table': [TABLES_DIR / name for name in required_tables],
    'Figure': [FIGURES_DIR / name for name in required_figures],
    'Shared artifact': required_shared_files
}

for category, paths in artifact_groups.items():

    for path in paths:

        available = path.is_file() and path.stat().st_size > 0

        verification.append({
            'Type': category,
            'Filename': path.name,
            'Status': 'Available' if available else 'Missing'
        })

verification_df = pd.DataFrame(verification)

display(verification_df)

verification_df.to_csv(
    TABLES_DIR / 'data_preparation_artifact_verification.csv',
    index=False
)

assert verification_df['Status'].eq('Available').all(), \
    "Some data preparation artifacts are missing"

# Save the data preparation summary

summary = f"""# Notebook 02 - Data Preparation

## Leakage Prevention
- Original training comments: {len(train_df):,}
- Overlapping training records removed: {len(removed_df)}
- Development pool: {len(development_df):,}
- Normalized duplicate groups are preserved within partitions.
- Original text and annotations remain unchanged.
- All 21 partition integrity checks passed.

## Final Data Partitions
- Full training: {len(train_full_df):,}
- Validation: {len(val_df):,}
- Official labelled test: {len(test_valid_df):,}
- Representative subset: {len(train_subset_df):,}
- Toxicity-enriched subset: {len(train_enriched_df):,}

## Splitting Strategy
Group-aware iterative multilabel stratification with an
80:20 train-validation ratio and random seed 42.

The maximum training-validation label prevalence difference
is approximately 0.0116 percentage points.

## Representative Subsampling
The primary training subset contains 59,997 comments,
retaining approximately 47.01% of the full training data.

The maximum label prevalence difference is approximately
0.0047 percentage points.

Rare-label positive counts:
- Severe toxic: 599
- Threat: 179
- Identity hate: 527

## Alternative Sampling
The toxicity-enriched subset contains 59,997 comments
and retains all positive training examples.

It is reserved for an optional controlled sampling experiment.
Its altered label prevalence must be considered during evaluation.

## Transformer Tokenization
- Models: DistilBERT and RoBERTa.
- Initial maximum sequence length: 256 tokens.
- Dynamic batch-level padding.
- Right-side truncation.
- Exact pretrained checkpoint revisions recorded.

Representative subset truncation rates:
- DistilBERT: 6.86%
- RoBERTa: 6.92%

Aggregate content token retention:
- DistilBERT: 82.38%
- RoBERTa: 81.86%

## Limitations
- Representative subsampling reduces rare-label support.
- Severe toxic and threat comments exhibit substantial token loss.
- Normalized duplicate detection does not identify all semantic
  near-duplicates.
- Historical exposure to official benchmark results is acknowledged.

## Planned Experiments
- Standard BCE, weighted BCE and focal loss.
- Representative versus enriched sampling.
- Subsample versus full-data training.
- Sequence length ablation at 384 or 512 tokens, subject to resources.
- Validation-based model selection and per-label threshold tuning.

## Evaluation Protocol
Validation and official test partitions remain unchanged.
Official test performance will not be used for model selection,
hyperparameter optimization or threshold tuning.

## Reproducibility
Prepared datasets, partition IDs, group assignments,
dataset fingerprints and tokenizer revisions are recorded.

## Conclusion
Data preparation is complete. All prepared datasets have
been saved and verified. No models have been trained.

## Next Stage
Notebook 03 - Classical Machine Learning.
"""

(RESULTS_DIR / 'summary.md').write_text(
    summary, encoding='utf-8'
)

print("\nData preparation completion summary")
print("Analysis tables verified:", len(required_tables))
print("Figures verified:", len(required_figures))
print("Shared artifacts verified:", len(required_shared_files))
print("Total artifacts verified:", len(verification_df))
print("Missing artifacts:", verification_df['Status'].eq('Missing').sum())
print("Data preparation completed successfully.")